# Comparación de Resolución Temporal y Velocidades: MATLAB vs. Veraset

Este notebook realiza una comparación estadística cuantitativa entre el dataset controlado de **MATLAB** (alta frecuencia, muestreo a 1 Hz) y el dataset de producción de **Veraset** (baja frecuencia, datos de telefonía en la vida real).

El objetivo es identificar las diferencias en:
1. **Frecuencia temporal ($\Delta t$ entre pings).**
2. **Presencia y longitud de baches (signal loss gaps).**
3. **Distribución de velocidades (instantánea vs. calculada por Haversine).**

Estos hallazgos servirán para calibrar las matrices probabilísticas bajo un escenario de datos degradados similar a la producción.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Configuración de Rutas y Carga de Datos

In [2]:
# Rutas relativas del proyecto
PROJECT_ROOT = Path("../../")
GPS_DATA_DIR = PROJECT_ROOT / "Inputs" / "GPS User Data"

PATH_MATLAB = GPS_DATA_DIR / "Datos de MATLAB GPS.csv"
PATH_VERASET = GPS_DATA_DIR / "top_20users_ALL_DATES.parquet"

print("Cargando dataset de MATLAB...")
df_matlab = pd.read_csv(PATH_MATLAB, low_memory=False)
print(f"MATLAB cargado con {len(df_matlab)} filas.")

print("Cargando dataset de Veraset...")
df_veraset = pd.read_parquet(PATH_VERASET)
print(f"Veraset cargado con {len(df_veraset)} filas.")

Cargando dataset de MATLAB...
MATLAB cargado con 398043 filas.
Cargando dataset de Veraset...
Veraset cargado con 1259202 filas.


## 2. Análisis del Intervalo Temporal ($\Delta t$)

In [3]:
# Convertir a datetime
df_matlab['datetime'] = pd.to_datetime(df_matlab['Timestamp'])
df_veraset['datetime'] = pd.to_datetime(df_veraset['utc_timestamp'], unit='s')

# Ordenar cronológicamente por usuario/viaje
df_matlab = df_matlab.sort_values(by=['caid', 'num_trip', 'datetime'])
df_veraset = df_veraset.sort_values(by=['caid', 'datetime'])

# Calcular deltas de tiempo en segundos
df_matlab['delta_t'] = df_matlab.groupby(['caid', 'num_trip'])['datetime'].diff().dt.total_seconds()
df_veraset['delta_t'] = df_veraset.groupby('caid')['datetime'].diff().dt.total_seconds()

# Filtrar deltas nulos o simultáneos
df_matlab_clean = df_matlab[df_matlab['delta_t'] > 0]
df_veraset_clean = df_veraset[df_veraset['delta_t'] > 0]

print("=== DELTA T MATLAB ===")
print(df_matlab_clean['delta_t'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

print("\n=== DELTA T VERASET ===")
print(df_veraset_clean['delta_t'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

=== DELTA T MATLAB ===
count    344555.000000
mean         17.506038
std        2013.867204
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
90%           1.000000
95%           1.000000
99%           2.000000
max      430195.000000
Name: delta_t, dtype: float64

=== DELTA T VERASET ===
count    1.259182e+06
mean     1.146552e+02
std      2.205998e+03
min      1.000000e+00
25%      5.000000e+00
50%      1.000000e+01
75%      2.400000e+01
90%      9.100000e+01
95%      2.590000e+02
99%      1.195000e+03
max      1.025115e+06
Name: delta_t, dtype: float64


### Gráfica de Comparación de Intervalos Temporales ($\Delta t$)

La gráfica generada a partir de los datos completos se visualiza a continuación:

![Distribución Delta T](distribucion_delta_t.png)

## 3. Análisis de Baches (Signal Loss Gaps)

Evaluamos la proporción de pings que ocurren tras una desconexión de señal de corta, mediana o larga duración.

In [4]:
for threshold in [60, 300, 900]:
    gaps_m = df_matlab_clean[df_matlab_clean['delta_t'] >= threshold]
    gaps_v = df_veraset_clean[df_veraset_clean['delta_t'] >= threshold]
    
    pct_m = len(gaps_m) / len(df_matlab_clean) * 100
    pct_v = len(gaps_v) / len(df_veraset_clean) * 100
    
    print(f"Intervalos >= {threshold}s (gaps de {threshold//60} min):")
    print(f"  MATLAB: {len(gaps_m)} ({pct_m:.4f}%)")
    print(f"  VERASET: {len(gaps_v)} ({pct_v:.4f}%)\n")

Intervalos >= 60s (gaps de 1 min):
  MATLAB: 318 (0.0923%)
  VERASET: 175017 (13.8993%)

Intervalos >= 300s (gaps de 5 min):
  MATLAB: 121 (0.0351%)
  VERASET: 56193 (4.4627%)

Intervalos >= 900s (gaps de 15 min):
  MATLAB: 90 (0.0261%)
  VERASET: 17447 (1.3856%)


## 4. Análisis de la Distribución de Velocidades

En Veraset, estimamos la velocidad promedio del intervalo calculando la distancia geodésica de Haversine y dividiéndola entre el tiempo transcurrido, y la comparamos con la velocidad instantánea física reportada por MATLAB.

In [5]:
# Fórmula Haversine vectorizada
def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

# Shift de coordenadas previas
df_veraset['lon_prev'] = df_veraset.groupby('caid')['longitude'].shift(1)
df_veraset['lat_prev'] = df_veraset.groupby('caid')['latitude'].shift(1)

# Distancia y velocidad estimada
df_veraset['dist_km'] = haversine_np(df_veraset['longitude'], df_veraset['latitude'], df_veraset['lon_prev'], df_veraset['lat_prev'])
df_veraset['speed_est'] = (df_veraset['dist_km'] / (df_veraset['delta_t'] / 3600.0))

# Filtrar outliers aberrantes en velocidad estimada
df_veraset_speed_clean = df_veraset[(df_veraset['delta_t'] > 0) & (df_veraset['speed_est'] < 200)]

print("=== VELOCIDADES ===")
print("MATLAB instantánea:")
print(df_matlab['speed'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

print("\nVERASET estimada:")
print(df_veraset_speed_clean['speed_est'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

=== VELOCIDADES ===
MATLAB instantánea:
count    398043.000000
mean         17.165951
std          23.824477
min          -0.759600
25%           0.000000
50%           4.618800
75%          28.346400
90%          56.462400
95%          69.616800
max         149.907600
Name: speed, dtype: float64

VERASET estimada:
count    1.249763e+06
mean     1.912132e+01
std      3.255678e+01
min      0.000000e+00
25%      0.000000e+00
50%      6.593529e-01
75%      2.867799e+01
90%      6.323948e+01
95%      8.599510e+01
max      1.999969e+02
Name: speed_est, dtype: float64


### Gráfica de Comparación de Distribución de Velocidades

La gráfica generada a partir de los datos completos se visualiza a continuación:

![Distribución Velocidades](distribucion_velocidades.png)